# PERSUADE v8: Enhanced Span Ablation Analysis

**Cohort:** score_long (50/50/50 low/mid/high, all ≥778 tokens, all EXTENDED regime)

**Primary influence metrics:**
1. **ΔNLL** (ablation – baseline): "how much predictive support came from that span"
2. **Rank displacement**: baseline rank of true token vs ablated rank (mean + median)
3. **Normalized influence**: span ΔNLL / total ΔNLL (within-essay proportion)

**Secondary diagnostics** (demoted): top-k Jaccard, flip rate

**Dose response:** Run with span_size=128 and span_size=256

**Goal:** Measure influence geometry - whether influence is distributed vs localized across spans, and whether high-quality essays redistribute influence (after controlling for fluency/length)

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate bitsandbytes pandas numpy matplotlib seaborn tqdm statsmodels

In [ ]:
import json
import math
import os
import time
import random
from datetime import datetime
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from scipy import stats
import statsmodels.formula.api as smf

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================

# Paths
DRIVE_BASE = "/content/drive/MyDrive/LRTIA/Data/persuade_clean"
OUTPUT_BASE = "/content/drive/MyDrive/LRTIA/Results/Persuade"
COHORT_PATH = f"{DRIVE_BASE}/cohorts/persuade_score_long_cohort.jsonl"

EXPERIMENT = 'score_long_span_ablation_v2'

# Model
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
USE_4BIT = False  # False for A100, True for T4

# Context/scoring parameters
CONTEXT_LENGTH = 512  # EXTENDED regime burn-in
TOTAL_SCORE_TOKENS = 256  # tokens 512-768

# Stratified scoring regions
SCORE_REGIONS = {
    'score_early': (512, 576),   # first 64 tokens after context
    'score_mid': (576, 640),     # next 64 tokens
    'score_late': (640, 768),    # final 128 tokens
}

# Ablation span sizes to test (DOSE RESPONSE)
SPAN_SIZES = [128, 256]  # Primary: 128, Dose: 256

def get_disjoint_spans(context_length, span_size):
    """
    Create NON-OVERLAPPING spans that tile the context.
    
    For span_size=128: 4 disjoint regions of 128 tokens each
    For span_size=256: 2 disjoint regions of 256 tokens each
    
    Returns dict of {label: (start, end)}
    """
    n_regions = context_length // span_size
    
    if n_regions >= 4:
        # 4 disjoint regions: early, early_mid, late, immediate_pre_target
        return {
            'early': (0, span_size),
            'early_mid': (span_size, 2*span_size),
            'late': (context_length - 2*span_size, context_length - span_size),
            'immediate_pre_target': (context_length - span_size, context_length),
        }
    elif n_regions >= 2:
        # 2 disjoint regions: early, immediate_pre_target
        return {
            'early': (0, span_size),
            'immediate_pre_target': (context_length - span_size, context_length),
        }
    else:
        return {'full_context': (0, context_length)}

# Random ablation settings
N_RANDOM_SPANS = 10  # Number of random spans per essay

# Secondary metrics
TOP_K = 10

# RANDOM SEED - ensures reproducibility of random spans
RANDOM_SEED = 42

print(f"Experiment: {EXPERIMENT}")
print(f"Cohort: {COHORT_PATH}")
print(f"\nContext: {CONTEXT_LENGTH} tokens")
print(f"Scoring regions: {SCORE_REGIONS}")
print(f"\nSpan sizes (dose response): {SPAN_SIZES}")
print(f"Random spans per essay: {N_RANDOM_SPANS}")
print(f"\n*** RANDOM_SEED = {RANDOM_SEED} (ensures reproducibility) ***")

# Print exact span boundaries for each span_size
print("\n" + "="*70)
print("SPAN BOUNDARIES (disjoint by construction)")
print("="*70)
for span_size in SPAN_SIZES:
    spans = get_disjoint_spans(CONTEXT_LENGTH, span_size)
    print(f"\nspan_size = {span_size}:")
    print(f"  {'Label':<25} {'Start':>8} {'End':>8} {'Size':>8}")
    print(f"  {'-'*55}")
    for label, (start, end) in sorted(spans.items(), key=lambda x: x[1][0]):
        print(f"  {label:<25} {start:>8} {end:>8} {end-start:>8}")
    print(f"  All spans disjoint: YES (by construction)")

## 1. Load Model and Data

In [ ]:
# Load tokenizer
print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
# Load model
print(f"Loading model: {MODEL_NAME}")

if USE_4BIT:
    print("  Using 4-bit quantization (T4 mode)")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, quantization_config=bnb_config, device_map="auto"
    )
else:
    print("  Using float16 (A100 mode)")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map="auto"
    )

model.eval()
print("Model loaded")

In [ ]:
# Load cohort
def load_cohort(path):
    records = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

cohort = load_cohort(COHORT_PATH)
print(f"Loaded {len(cohort)} essays")

# Verify cohort
df_cohort = pd.DataFrame(cohort)
print(f"\nScore bin distribution:")
print(df_cohort['score_bin'].value_counts())

## 2. Core Functions

In [ ]:
@torch.no_grad()
def get_logits(token_ids):
    """Get logits for all positions in token_ids."""
    input_ids = torch.tensor([token_ids], device=model.device)
    outputs = model(input_ids)
    return outputs.logits[0]


def compute_metrics_for_region(logits, token_ids, target_start, target_end):
    """Compute primary influence metrics for a scoring region."""
    if target_end > len(token_ids):
        target_end = len(token_ids)
    if target_start >= target_end - 1:
        return {'nlls': [], 'ranks': [], 'top1_preds': [], 'topk_sets': [], 'true_tokens': []}
    
    nlls = []
    ranks = []
    top1_preds = []
    topk_sets = []
    true_tokens = []
    
    for i in range(target_start, target_end - 1):
        if i >= logits.shape[0]:
            break
        
        true_token = token_ids[i + 1]
        true_tokens.append(true_token)
        
        log_probs = torch.log_softmax(logits[i], dim=-1)
        nll = -log_probs[true_token].item()
        nlls.append(nll)
        
        sorted_indices = logits[i].argsort(descending=True)
        rank = (sorted_indices == true_token).nonzero(as_tuple=True)[0].item()
        ranks.append(rank)
        
        top1 = logits[i].argmax().item()
        top1_preds.append(top1)
        
        topk = set(logits[i].topk(TOP_K).indices.tolist())
        topk_sets.append(topk)
    
    return {
        'nlls': nlls,
        'ranks': ranks,
        'top1_preds': top1_preds,
        'topk_sets': topk_sets,
        'true_tokens': true_tokens,
    }


def compute_comparison_metrics(baseline, ablated):
    """Compare baseline vs ablated predictions."""
    if not baseline['nlls'] or not ablated['nlls']:
        return None
    
    n = min(len(baseline['nlls']), len(ablated['nlls']))
    
    baseline_nll_mean = np.mean(baseline['nlls'][:n])
    ablated_nll_mean = np.mean(ablated['nlls'][:n])
    delta_nll = ablated_nll_mean - baseline_nll_mean
    
    baseline_rank_mean = np.mean(baseline['ranks'][:n])
    ablated_rank_mean = np.mean(ablated['ranks'][:n])
    rank_displacement = ablated_rank_mean - baseline_rank_mean
    
    rank_shifts = [ablated['ranks'][i] - baseline['ranks'][i] for i in range(n)]
    rank_displacement_median = np.median(rank_shifts)
    
    n_flips = sum(1 for i in range(n) if baseline['top1_preds'][i] != ablated['top1_preds'][i])
    flip_rate = n_flips / n
    
    topk_jaccard_changes = []
    for i in range(n):
        b_set = baseline['topk_sets'][i]
        a_set = ablated['topk_sets'][i]
        intersection = len(b_set & a_set)
        union = len(b_set | a_set)
        jaccard_sim = intersection / union if union > 0 else 1.0
        topk_jaccard_changes.append(1.0 - jaccard_sim)
    topk_change = np.mean(topk_jaccard_changes)
    
    return {
        'delta_nll': delta_nll,
        'rank_displacement_mean': rank_displacement,
        'rank_displacement_median': rank_displacement_median,
        'baseline_nll': baseline_nll_mean,
        'ablated_nll': ablated_nll_mean,
        'baseline_rank_mean': baseline_rank_mean,
        'ablated_rank_mean': ablated_rank_mean,
        'flip_rate': flip_rate,
        'topk_change': topk_change,
        'n_tokens': n,
    }


def ablate_span_deletion(token_ids, span_start, span_end):
    """Delete span and concatenate remaining tokens."""
    return token_ids[:span_start] + token_ids[span_end:]


print("Core functions defined")

In [ ]:
def run_span_ablation_for_essay(token_ids, essay_id, span_size, rng):
    """Run span ablation analysis for a single essay with stratified scoring."""
    n_tokens = len(token_ids)
    min_required = CONTEXT_LENGTH + TOTAL_SCORE_TOKENS
    if n_tokens < min_required:
        return []
    
    full_seq = token_ids[:min_required]
    baseline_logits = get_logits(full_seq)
    
    baseline_by_region = {}
    for region_name, (reg_start, reg_end) in SCORE_REGIONS.items():
        baseline_by_region[region_name] = compute_metrics_for_region(
            baseline_logits, full_seq, reg_start, reg_end
        )
    
    results = []
    
    # Get DISJOINT spans for this span_size
    span_configs = get_disjoint_spans(CONTEXT_LENGTH, span_size)
    
    # Fixed-position spans (DISJOINT)
    for span_label, (span_start, span_end) in span_configs.items():
        actual_span_size = span_end - span_start
        
        ablated_context = ablate_span_deletion(full_seq[:CONTEXT_LENGTH], span_start, span_end)
        ablated_full = ablated_context + full_seq[CONTEXT_LENGTH:]
        ablated_logits = get_logits(ablated_full)
        
        shift = CONTEXT_LENGTH - len(ablated_context)
        
        for region_name, (reg_start, reg_end) in SCORE_REGIONS.items():
            adj_start = reg_start - shift
            adj_end = reg_end - shift
            
            ablated_metrics = compute_metrics_for_region(
                ablated_logits, ablated_full, adj_start, adj_end
            )
            
            comparison = compute_comparison_metrics(
                baseline_by_region[region_name],
                ablated_metrics
            )
            
            if comparison:
                # Add per-token influence metric
                delta_nll_per_token = comparison['delta_nll'] / actual_span_size if actual_span_size > 0 else 0
                
                results.append({
                    'essay_id': essay_id,
                    'span_label': span_label,
                    'span_size': actual_span_size,
                    'span_start_token': span_start,
                    'span_end_token': span_end,
                    'score_region': region_name,
                    'delta_nll_per_token': delta_nll_per_token,
                    **comparison,
                })
    
    # Random spans (N_RANDOM_SPANS per essay, averaged)
    random_results_by_region = defaultdict(list)
    
    for rand_idx in range(N_RANDOM_SPANS):
        max_start = CONTEXT_LENGTH - span_size
        if max_start <= 0:
            continue
        span_start = rng.randint(0, max_start)
        span_end = span_start + span_size
        
        ablated_context = ablate_span_deletion(full_seq[:CONTEXT_LENGTH], span_start, span_end)
        ablated_full = ablated_context + full_seq[CONTEXT_LENGTH:]
        ablated_logits = get_logits(ablated_full)
        
        shift = CONTEXT_LENGTH - len(ablated_context)
        
        for region_name, (reg_start, reg_end) in SCORE_REGIONS.items():
            adj_start = reg_start - shift
            adj_end = reg_end - shift
            
            ablated_metrics = compute_metrics_for_region(
                ablated_logits, ablated_full, adj_start, adj_end
            )
            
            comparison = compute_comparison_metrics(
                baseline_by_region[region_name],
                ablated_metrics
            )
            
            if comparison:
                random_results_by_region[region_name].append(comparison)
    
    # Average random results within essay
    for region_name, comparisons in random_results_by_region.items():
        if comparisons:
            mean_delta_nll = np.mean([c['delta_nll'] for c in comparisons])
            delta_nll_per_token = mean_delta_nll / span_size if span_size > 0 else 0
            
            results.append({
                'essay_id': essay_id,
                'span_label': 'random',
                'span_size': span_size,
                'span_start_token': np.nan,
                'span_end_token': np.nan,
                'score_region': region_name,
                'delta_nll': mean_delta_nll,
                'delta_nll_per_token': delta_nll_per_token,
                'rank_displacement_mean': np.mean([c['rank_displacement_mean'] for c in comparisons]),
                'rank_displacement_median': np.median([c['rank_displacement_median'] for c in comparisons]),
                'baseline_nll': np.mean([c['baseline_nll'] for c in comparisons]),
                'ablated_nll': np.mean([c['ablated_nll'] for c in comparisons]),
                'baseline_rank_mean': np.mean([c['baseline_rank_mean'] for c in comparisons]),
                'ablated_rank_mean': np.mean([c['ablated_rank_mean'] for c in comparisons]),
                'flip_rate': np.mean([c['flip_rate'] for c in comparisons]),
                'topk_change': np.mean([c['topk_change'] for c in comparisons]),
                'n_tokens': comparisons[0]['n_tokens'],
                'n_random_samples': len(comparisons),
            })
    
    return results


print("Span ablation function defined (with disjoint spans and per-token metric)")

## 3. Run Span Ablation (Dose Response: 128 and 256)

In [ ]:
# Pre-tokenize all essays
essay_tokens = {}
for essay in cohort:
    token_ids = tokenizer.encode(essay['text'], add_special_tokens=False)
    essay_tokens[essay['essay_id']] = token_ids

print(f"Tokenized {len(essay_tokens)} essays")

lengths = [len(t) for t in essay_tokens.values()]
print(f"Token lengths: min={min(lengths)}, median={np.median(lengths):.0f}, max={max(lengths)}")

In [ ]:
# SANITY CHECK: Confirm random seed is set and reproducible
print("="*80)
print("SANITY CHECK: Random seed confirmation")
print("="*80)
print(f"\nRANDOM_SEED = {RANDOM_SEED}")

# Test reproducibility
test_rng1 = random.Random(RANDOM_SEED)
test_rng2 = random.Random(RANDOM_SEED)
test_vals1 = [test_rng1.randint(0, 100) for _ in range(5)]
test_vals2 = [test_rng2.randint(0, 100) for _ in range(5)]
print(f"Test sequence 1: {test_vals1}")
print(f"Test sequence 2: {test_vals2}")
print(f"Sequences match: {test_vals1 == test_vals2}")
assert test_vals1 == test_vals2, "Random seed not reproducible!"
print("\n*** Random spans will be reproducible across runs ***")

In [ ]:
# Process all essays with DOSE RESPONSE
all_results = []
start_time = time.time()

for span_size in SPAN_SIZES:
    print(f"\n{'='*60}")
    print(f"DOSE: span_size = {span_size} tokens")
    print(f"{'='*60}")
    
    # Reset RNG for each span size (ensures reproducibility)
    rng = random.Random(RANDOM_SEED)
    
    # Get disjoint spans for this span_size
    span_configs = get_disjoint_spans(CONTEXT_LENGTH, span_size)
    n_fixed_spans = len(span_configs)
    n_spans = n_fixed_spans + 1  # +1 for random
    n_regions = len(SCORE_REGIONS)
    expected_rows = len(cohort) * n_spans * n_regions
    
    print(f"Disjoint spans for this size: {list(span_configs.keys())}")
    print(f"Expected: {len(cohort)} essays × {n_spans} spans × {n_regions} regions = {expected_rows} rows")
    
    for essay in tqdm(cohort, desc=f"Span size {span_size}"):
        essay_id = essay['essay_id']
        token_ids = essay_tokens[essay_id]
        
        results = run_span_ablation_for_essay(token_ids, essay_id, span_size, rng)
        
        for r in results:
            r['score'] = essay.get('score')
            r['score_bin'] = essay.get('score_bin')
            r['grade'] = essay.get('grade')
            r['token_count'] = len(token_ids)
        
        all_results.extend(results)

elapsed = time.time() - start_time
print(f"\nDone in {elapsed:.1f}s ({elapsed/len(cohort)/len(SPAN_SIZES):.2f}s/essay/dose)")
print(f"Total result rows: {len(all_results)}")

In [ ]:
# Create DataFrame
df = pd.DataFrame(all_results)

print(f"Results shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSpan size × Span label counts:")
print(pd.crosstab(df['span_size'], df['span_label']))

df.head(10)

## 4. Sanity Checks

In [ ]:
# SANITY CHECK 1: Baseline and ablated NLL ranges
print("="*80)
print("SANITY CHECK 1: NLL Ranges (ensure deltas aren't saturating or tiny)")
print("="*80)

for span_size in SPAN_SIZES:
    subset = df[df['span_size'] == span_size]
    print(f"\n--- Span size {span_size} ---")
    print(f"Baseline NLL: mean={subset['baseline_nll'].mean():.3f}, "
          f"std={subset['baseline_nll'].std():.3f}, "
          f"range=[{subset['baseline_nll'].min():.3f}, {subset['baseline_nll'].max():.3f}]")
    print(f"Ablated NLL:  mean={subset['ablated_nll'].mean():.3f}, "
          f"std={subset['ablated_nll'].std():.3f}, "
          f"range=[{subset['ablated_nll'].min():.3f}, {subset['ablated_nll'].max():.3f}]")
    print(f"Delta NLL:    mean={subset['delta_nll'].mean():.4f}, "
          f"std={subset['delta_nll'].std():.4f}, "
          f"range=[{subset['delta_nll'].min():.4f}, {subset['delta_nll'].max():.4f}]")
    print(f"Rank displacement (mean): mean={subset['rank_displacement_mean'].mean():.1f}, "
          f"range=[{subset['rank_displacement_mean'].min():.1f}, {subset['rank_displacement_mean'].max():.1f}]")

In [ ]:
# SANITY CHECK 2: Delta distributions by span
print("\n" + "="*80)
print("SANITY CHECK 2: Mean ΔNLL by Span Label (should be positive, larger for boundary-adjacent)")
print("="*80)

for span_size in SPAN_SIZES:
    print(f"\n--- Span size {span_size} ---")
    subset = df[df['span_size'] == span_size]
    summary = subset.groupby('span_label')['delta_nll'].agg(['mean', 'std', 'count'])
    print(summary.round(4))

In [ ]:
# SANITY CHECK 3: Total ΔNLL per essay distribution
print("\n" + "="*80)
print("SANITY CHECK 3: Total ΔNLL per Essay (for normalization meaningfulness)")
print("="*80)

# Compute total ΔNLL per essay (sum across fixed spans, excluding random)
df_128 = df[df['span_size'] == 128]
fixed_spans = df_128[df_128['span_label'] != 'random']

total_dnll_per_essay = fixed_spans.groupby(['essay_id', 'score_region'])['delta_nll'].sum().reset_index()
total_dnll_per_essay.columns = ['essay_id', 'score_region', 'total_delta_nll']

print(f"\nTotal ΔNLL per essay (span_size=128):")
print(f"  Mean:   {total_dnll_per_essay['total_delta_nll'].mean():.4f}")
print(f"  Median: {total_dnll_per_essay['total_delta_nll'].median():.4f}")
print(f"  Std:    {total_dnll_per_essay['total_delta_nll'].std():.4f}")
print(f"  Min:    {total_dnll_per_essay['total_delta_nll'].min():.4f}")
print(f"  Max:    {total_dnll_per_essay['total_delta_nll'].max():.4f}")

# Check for essays with very low total influence (normalization may be unstable)
low_influence = total_dnll_per_essay[total_dnll_per_essay['total_delta_nll'] < 0.01]
print(f"\n  Essays with total ΔNLL < 0.01: {len(low_influence)} (may have unstable normalization)")

# Distribution by score region
print(f"\nBy score region:")
for region in ['score_early', 'score_mid', 'score_late']:
    region_data = total_dnll_per_essay[total_dnll_per_essay['score_region'] == region]
    print(f"  {region}: mean={region_data['total_delta_nll'].mean():.4f}, "
          f"std={region_data['total_delta_nll'].std():.4f}")

## 5. Build Influence Profiles

In [ ]:
# Define ordering
GROUP_ORDER = ['low', 'mid', 'high']
# Note: span labels vary by span_size
# span_size=128: early, early_mid, late, immediate_pre_target, random
# span_size=256: early, immediate_pre_target, random
SPAN_ORDER_128 = ['early', 'early_mid', 'late', 'immediate_pre_target', 'random']
SPAN_ORDER_256 = ['early', 'immediate_pre_target', 'random']
SPAN_ORDER = SPAN_ORDER_128  # Default for display
REGION_ORDER = ['score_early', 'score_mid', 'score_late']

df['score_bin'] = pd.Categorical(df['score_bin'], categories=GROUP_ORDER, ordered=True)
df['score_region'] = pd.Categorical(df['score_region'], categories=REGION_ORDER, ordered=True)

In [ ]:
# Compute NORMALIZED influence profiles per essay
def compute_normalized_influence(group):
    """Normalize delta_nll within an essay (for a given span_size and score_region)."""
    fixed_spans = group[group['span_label'] != 'random']
    total_influence = fixed_spans['delta_nll'].abs().sum()
    
    group = group.copy()
    if total_influence > 0:
        group['delta_nll_normalized'] = group['delta_nll'] / total_influence
    else:
        group['delta_nll_normalized'] = 0.0
    
    return group

df = df.groupby(['essay_id', 'span_size', 'score_region'], group_keys=False).apply(
    compute_normalized_influence
)

print("Normalized influence computed")
print(f"\nSample normalized values (span_size=128, score_early):")
sample = df[(df['span_size'] == 128) & (df['score_region'] == 'score_early')]
print(sample.groupby('span_label')['delta_nll_normalized'].mean().round(4))

## 6. Group Summaries

In [ ]:
# Summary by score_bin × span_label × score_region × span_size
print("="*80)
print("PRIMARY METRIC: ΔNLL BY SCORE_BIN × SPAN_LABEL × SCORE_REGION")
print("="*80)

summary_rows = []

for span_size in SPAN_SIZES:
    # Get the valid span labels for this span_size
    span_configs = get_disjoint_spans(CONTEXT_LENGTH, span_size)
    valid_labels = list(span_configs.keys()) + ['random']
    
    for score_bin in GROUP_ORDER:
        for span_label in valid_labels:
            for region in REGION_ORDER:
                subset = df[(df['span_size'] == span_size) &
                           (df['score_bin'] == score_bin) & 
                           (df['span_label'] == span_label) & 
                           (df['score_region'] == region)]
                if len(subset) == 0:
                    continue
                
                summary_rows.append({
                    'span_size': span_size,
                    'score_bin': score_bin,
                    'span_label': span_label,
                    'score_region': region,
                    'n': len(subset),
                    'delta_nll_mean': subset['delta_nll'].mean(),
                    'delta_nll_sem': subset['delta_nll'].sem(),
                    'delta_nll_ci95': 1.96 * subset['delta_nll'].sem(),
                    'delta_nll_per_token_mean': subset['delta_nll_per_token'].mean(),
                    'delta_nll_normalized_mean': subset['delta_nll_normalized'].mean(),
                    'rank_displacement_mean': subset['rank_displacement_mean'].mean(),
                    'rank_displacement_median': subset['rank_displacement_median'].median(),
                    'baseline_nll_mean': subset['baseline_nll'].mean(),
                    'ablated_nll_mean': subset['ablated_nll'].mean(),
                    'flip_rate_mean': subset['flip_rate'].mean(),
                    'topk_change_mean': subset['topk_change'].mean(),
                })

df_summary = pd.DataFrame(summary_rows)
print(f"Summary table: {len(df_summary)} rows")

In [ ]:
# KEY TABLE: Pivoted view for span_size=128
print("\n" + "="*80)
print("PIVOT: Mean ΔNLL (span_size=128)")
print("="*80)

df_128 = df[df['span_size'] == 128]

for score_bin in GROUP_ORDER:
    print(f"\n--- Score bin: {score_bin} ---")
    subset = df_128[df_128['score_bin'] == score_bin]
    pivot = subset.pivot_table(
        values='delta_nll',
        index='span_label',
        columns='score_region',
        aggfunc='mean'
    )
    if REGION_ORDER[0] in pivot.columns:
        pivot = pivot[REGION_ORDER]
    print(pivot.round(4))

In [ ]:
# NORMALIZED influence profiles
print("\n" + "="*80)
print("PIVOT: Normalized Influence (within-essay proportions, span_size=128)")
print("="*80)

for score_bin in GROUP_ORDER:
    print(f"\n--- Score bin: {score_bin} ---")
    subset = df_128[(df_128['score_bin'] == score_bin) & (df_128['span_label'] != 'random')]
    pivot = subset.pivot_table(
        values='delta_nll_normalized',
        index='span_label',
        columns='score_region',
        aggfunc='mean'
    )
    if REGION_ORDER[0] in pivot.columns:
        pivot = pivot[REGION_ORDER]
    print(pivot.round(4))

In [ ]:
# DOSE RESPONSE: Compare span_size 128 vs 256
print("\n" + "="*80)
print("DOSE RESPONSE: Span Size 128 vs 256")
print("="*80)
print("Does the influence profile SHAPE change with larger ablation?")

for region in REGION_ORDER:
    print(f"\n--- {region} ---")
    print(f"{'Span':<25} {'Size 128':>12} {'Size 256':>12} {'Ratio':>10}")
    print("-"*65)
    
    for span_label in SPAN_ORDER[:-1]:
        val_128 = df_summary[(df_summary['span_size'] == 128) & 
                            (df_summary['span_label'] == span_label) & 
                            (df_summary['score_region'] == region)]['delta_nll_mean'].mean()
        val_256 = df_summary[(df_summary['span_size'] == 256) & 
                            (df_summary['span_label'] == span_label) & 
                            (df_summary['score_region'] == region)]['delta_nll_mean'].mean()
        ratio = val_256 / val_128 if val_128 != 0 else np.nan
        print(f"{span_label:<25} {val_128:>12.4f} {val_256:>12.4f} {ratio:>10.2f}")

## 7. Statistical Tests

In [ ]:
# Standardize controls
df['token_count_z'] = (df['token_count'] - df['token_count'].mean()) / df['token_count'].std()
df['baseline_nll_z'] = (df['baseline_nll'] - df['baseline_nll'].mean()) / df['baseline_nll'].std()

df_128 = df[df['span_size'] == 128].copy()

print("="*80)
print("REGRESSION: delta_nll ~ span + region + score_bin + controls + interactions")
print("="*80)

formula = ('delta_nll ~ C(span_label) + C(score_region) + C(score_bin) + '
           'C(span_label):C(score_region) + C(span_label):C(score_bin) + '
           'token_count_z + baseline_nll_z')

model_full = smf.ols(formula, data=df_128).fit()

print(f"\nFormula: {formula}")
print(f"R²: {model_full.rsquared:.4f}, Adj R²: {model_full.rsquared_adj:.4f}, n={int(model_full.nobs)}")

In [ ]:
# Key coefficients: span effects
print("\n" + "="*80)
print("SPAN EFFECTS (vs early)")
print("="*80)

span_params = [p for p in model_full.params.index if 'span_label' in p and ':' not in p]
for param in sorted(span_params):
    coef = model_full.params[param]
    pval = model_full.pvalues[param]
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

In [ ]:
# Key test: span × region interaction
print("\n" + "="*80)
print("SPAN × REGION INTERACTION (does span influence vary by distance to target?)")
print("="*80)

interaction_params = [p for p in model_full.params.index if 'span_label' in p and 'score_region' in p]
print(f"\n{len(interaction_params)} interaction terms:")
for param in sorted(interaction_params)[:10]:
    coef = model_full.params[param]
    pval = model_full.pvalues[param]
    sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
    print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

In [ ]:
# Test: Does score_bin affect normalized influence profiles?
print("\n" + "="*80)
print("GROUP ANALYSIS: Do score bins differ in WHERE influence comes from?")
print("="*80)

df_norm = df_128[df_128['span_label'] != 'random'].copy()

formula_norm = ('delta_nll_normalized ~ C(span_label) + C(score_region) + C(score_bin) + '
                'C(span_label):C(score_bin)')

model_norm = smf.ols(formula_norm, data=df_norm).fit()

print(f"\nNormalized influence model:")
print(f"R²: {model_norm.rsquared:.4f}")

score_span_params = [p for p in model_norm.params.index if 'score_bin' in p and 'span_label' in p]
print(f"\nScore bin × Span interactions (do groups use context differently?):")
sig_count = 0
for param in sorted(score_span_params):
    coef = model_norm.params[param]
    pval = model_norm.pvalues[param]
    if pval < 0.05:
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*"
        print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")
        sig_count += 1

if sig_count == 0:
    print("  No significant interactions (groups use context similarly)")

In [ ]:
# Raw ΔNLL comparison controlling for fluency
print("\n" + "="*80)
print("GROUP ANALYSIS: Raw ΔNLL (absolute influence) controlling for fluency")
print("="*80)

formula_raw = 'delta_nll ~ C(score_bin) + baseline_nll_z + token_count_z'
model_raw = smf.ols(formula_raw, data=df_128).fit()

print(f"\nModel: {formula_raw}")
print(f"R²: {model_raw.rsquared:.4f}")
print(f"\nScore bin effects (controlling for fluency):")
for param in ['C(score_bin)[T.mid]', 'C(score_bin)[T.high]']:
    if param in model_raw.params:
        coef = model_raw.params[param]
        pval = model_raw.pvalues[param]
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

print(f"\nFluency control (baseline_nll_z):")
print(f"  β={model_raw.params['baseline_nll_z']:+.4f}, p={model_raw.pvalues['baseline_nll_z']:.4f}")

In [ ]:
# Rank displacement regression
print("\n" + "="*80)
print("GROUP ANALYSIS: Rank Displacement controlling for fluency")
print("="*80)

formula_rank = 'rank_displacement_mean ~ C(score_bin) + baseline_nll_z + token_count_z'
model_rank = smf.ols(formula_rank, data=df_128).fit()

print(f"\nModel: {formula_rank}")
print(f"R²: {model_rank.rsquared:.4f}")
print(f"\nScore bin effects:")
for param in ['C(score_bin)[T.mid]', 'C(score_bin)[T.high]']:
    if param in model_rank.params:
        coef = model_rank.params[param]
        pval = model_rank.pvalues[param]
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        print(f"  {param}: β={coef:+.4f}, p={pval:.4f} {sig}")

## 8. Visualizations

In [ ]:
# Color schemes
SCORE_COLORS = {'low': '#e74c3c', 'mid': '#f39c12', 'high': '#2ecc71'}
REGION_COLORS = {'score_early': '#3498db', 'score_mid': '#9b59b6', 'score_late': '#1abc9c'}

In [ ]:
# Create output directory
output_dir = Path(OUTPUT_BASE) / EXPERIMENT
output_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
# KEY PLOT 1: Aggregate heatmaps (raw + normalized)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Raw ΔNLL
ax = axes[0]
pivot_raw = df_128.pivot_table(
    values='delta_nll',
    index='span_label',
    columns='score_region',
    aggfunc='mean'
)[REGION_ORDER]

sns.heatmap(pivot_raw, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'ΔNLL'})
ax.set_title('Raw ΔNLL: Span × Region (Aggregated)\n(Higher = more influence)', 
             fontsize=11, fontweight='bold')
ax.set_xlabel('Scoring Region')
ax.set_ylabel('Ablated Span')

# Normalized
ax = axes[1]
pivot_norm = df_128[df_128['span_label'] != 'random'].pivot_table(
    values='delta_nll_normalized',
    index='span_label',
    columns='score_region',
    aggfunc='mean'
)[REGION_ORDER]

sns.heatmap(pivot_norm, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': 'Normalized Influence'})
ax.set_title('Normalized Influence (Aggregated)\n(Within-essay proportions)', 
             fontsize=11, fontweight='bold')
ax.set_xlabel('Scoring Region')
ax.set_ylabel('Ablated Span')

plt.tight_layout()
plt.savefig(output_dir / 'influence_heatmaps.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# KEY PLOT 2: Heatmaps SPLIT BY SCORE BIN
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

for col_idx, score_bin in enumerate(GROUP_ORDER):
    subset = df_128[df_128['score_bin'] == score_bin]
    
    # Raw ΔNLL (top row)
    ax = axes[0, col_idx]
    pivot = subset.pivot_table(
        values='delta_nll',
        index='span_label',
        columns='score_region',
        aggfunc='mean'
    )
    if REGION_ORDER[0] in pivot.columns:
        pivot = pivot[REGION_ORDER]
    
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
                cbar_kws={'label': 'ΔNLL'}, vmin=0, vmax=pivot_raw.max().max())
    ax.set_title(f'Raw ΔNLL: {score_bin.upper()}', fontsize=12, fontweight='bold',
                 color=SCORE_COLORS[score_bin])
    ax.set_xlabel('Scoring Region')
    ax.set_ylabel('Ablated Span' if col_idx == 0 else '')
    
    # Normalized (bottom row)
    ax = axes[1, col_idx]
    subset_norm = subset[subset['span_label'] != 'random']
    pivot = subset_norm.pivot_table(
        values='delta_nll_normalized',
        index='span_label',
        columns='score_region',
        aggfunc='mean'
    )
    if REGION_ORDER[0] in pivot.columns:
        pivot = pivot[REGION_ORDER]
    
    sns.heatmap(pivot, annot=True, fmt='.3f', cmap='YlOrRd', ax=ax,
                cbar_kws={'label': 'Normalized'}, vmin=0, vmax=pivot_norm.max().max())
    ax.set_title(f'Normalized: {score_bin.upper()}', fontsize=12, fontweight='bold',
                 color=SCORE_COLORS[score_bin])
    ax.set_xlabel('Scoring Region')
    ax.set_ylabel('Ablated Span' if col_idx == 0 else '')

fig.suptitle('Influence Heatmaps by Score Bin (span_size=128)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(output_dir / 'influence_heatmaps_by_scorebin.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# KEY PLOT 3: Influence profiles by score bin (bar chart)
# Use only fixed spans (exclude random) for span_size=128
FIXED_SPANS_128 = ['early', 'early_mid', 'late', 'immediate_pre_target']

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)

x_positions = np.arange(len(FIXED_SPANS_128))
width = 0.25

for ax_idx, region in enumerate(REGION_ORDER):
    ax = axes[ax_idx]
    region_data = df_128[(df_128['score_region'] == region) & (df_128['span_label'].isin(FIXED_SPANS_128))]
    
    for i, score_bin in enumerate(GROUP_ORDER):
        means = []
        cis = []
        for span in FIXED_SPANS_128:
            subset = region_data[(region_data['score_bin'] == score_bin) & 
                                (region_data['span_label'] == span)]
            if len(subset) > 0:
                means.append(subset['delta_nll'].mean())
                cis.append(1.96 * subset['delta_nll'].sem())
            else:
                means.append(np.nan)
                cis.append(np.nan)
        
        offset = (i - 1) * width
        ax.bar(x_positions + offset, means, width, yerr=cis,
               label=score_bin, color=SCORE_COLORS[score_bin], capsize=2, alpha=0.8)
    
    ax.set_title(f'{region}\n(tokens {SCORE_REGIONS[region][0]}-{SCORE_REGIONS[region][1]})', 
                 fontsize=11, fontweight='bold')
    ax.set_xticks(x_positions)
    ax.set_xticklabels(['early\n[0-128)', 'early_mid\n[128-256)', 'late\n[256-384)', 'immed_pre\n[384-512)'],
                       fontsize=8)
    ax.grid(True, alpha=0.3, axis='y')
    
    if ax_idx == 0:
        ax.set_ylabel('ΔNLL (ablated - baseline)', fontsize=11)
    if ax_idx == 1:
        ax.legend(title='Score Bin', loc='upper left', fontsize=9)

fig.suptitle('Span Influence by Score Bin and Scoring Region (span_size=128)\n'
             '(Higher ΔNLL = more predictive support from that span)', 
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(output_dir / 'influence_by_group.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# KEY PLOT 4: Decay analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# immediate_pre_target decay across scoring regions
ax = axes[0]
imm_pre = df_128[df_128['span_label'] == 'immediate_pre_target']

for score_bin in GROUP_ORDER:
    means = []
    cis = []
    for region in REGION_ORDER:
        subset = imm_pre[(imm_pre['score_bin'] == score_bin) & (imm_pre['score_region'] == region)]
        if len(subset) > 0:
            means.append(subset['delta_nll'].mean())
            cis.append(1.96 * subset['delta_nll'].sem())
        else:
            means.append(np.nan)
            cis.append(np.nan)
    
    ax.errorbar(range(len(REGION_ORDER)), means, yerr=cis, marker='o', capsize=4,
                label=score_bin, color=SCORE_COLORS[score_bin], linewidth=2, markersize=8)

ax.set_xticks(range(len(REGION_ORDER)))
ax.set_xticklabels(['Early\n(512-576)', 'Mid\n(576-640)', 'Late\n(640-768)'])
ax.set_xlabel('Scoring Region (distance from ablated span)', fontsize=11)
ax.set_ylabel('ΔNLL', fontsize=11)
ax.set_title('Immediate Pre-Target Ablation:\nEffect Decay with Distance', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

# Rank displacement by span (for score_early region)
ax = axes[1]
score_early = df_128[(df_128['score_region'] == 'score_early') & (df_128['span_label'].isin(FIXED_SPANS_128))]

for score_bin in GROUP_ORDER:
    means = []
    for span in FIXED_SPANS_128:
        subset = score_early[(score_early['score_bin'] == score_bin) & 
                            (score_early['span_label'] == span)]
        if len(subset) > 0:
            means.append(subset['rank_displacement_mean'].mean())
        else:
            means.append(np.nan)
    
    ax.plot(range(len(FIXED_SPANS_128)), means, marker='s', label=score_bin, 
            color=SCORE_COLORS[score_bin], linewidth=2, markersize=8)

ax.set_xticks(range(len(FIXED_SPANS_128)))
ax.set_xticklabels(['early\n[0-128)', 'early_mid\n[128-256)', 'late\n[256-384)', 'immed_pre\n[384-512)'], 
                   rotation=45, ha='right', fontsize=9)
ax.set_xlabel('Ablated Span (left=distant, right=adjacent)', fontsize=11)
ax.set_ylabel('Rank Displacement (mean)', fontsize=11)
ax.set_title('score_early Region:\nRank Displacement by Span Distance', fontsize=12, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'decay_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# KEY PLOT 5: Dose response - compare span_size 128 vs 256
# Only compare spans that exist in BOTH sizes: early, immediate_pre_target
COMMON_SPANS = ['early', 'immediate_pre_target']

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Row 1: Raw ΔNLL
for ax_idx, region in enumerate(REGION_ORDER):
    ax = axes[0, ax_idx]
    
    for span_size, marker, ls in [(128, 'o', '-'), (256, 's', '--')]:
        subset = df[(df['span_size'] == span_size) & 
                   (df['score_region'] == region) & 
                   (df['span_label'].isin(COMMON_SPANS))]
        
        means = []
        for span in COMMON_SPANS:
            span_subset = subset[subset['span_label'] == span]
            if len(span_subset) > 0:
                means.append(span_subset['delta_nll'].mean())
            else:
                means.append(np.nan)
        
        ax.plot(range(len(COMMON_SPANS)), means, marker=marker, linestyle=ls, 
                label=f'span={span_size}', linewidth=2, markersize=8)
    
    ax.set_xticks(range(len(COMMON_SPANS)))
    ax.set_xticklabels(['early\n[0-span)', 'immed_pre\n[512-span, 512)'], fontsize=9)
    ax.set_title(f'{region}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    if ax_idx == 0:
        ax.set_ylabel('Raw ΔNLL', fontsize=11)
        ax.legend(fontsize=9)

# Row 2: Per-token ΔNLL (normalized by span size)
for ax_idx, region in enumerate(REGION_ORDER):
    ax = axes[1, ax_idx]
    
    for span_size, marker, ls in [(128, 'o', '-'), (256, 's', '--')]:
        subset = df[(df['span_size'] == span_size) & 
                   (df['score_region'] == region) & 
                   (df['span_label'].isin(COMMON_SPANS))]
        
        means = []
        for span in COMMON_SPANS:
            span_subset = subset[subset['span_label'] == span]
            if len(span_subset) > 0:
                means.append(span_subset['delta_nll_per_token'].mean())
            else:
                means.append(np.nan)
        
        ax.plot(range(len(COMMON_SPANS)), means, marker=marker, linestyle=ls, 
                label=f'span={span_size}', linewidth=2, markersize=8)
    
    ax.set_xticks(range(len(COMMON_SPANS)))
    ax.set_xticklabels(['early', 'immed_pre'], fontsize=9)
    ax.set_title(f'{region}', fontsize=11, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    if ax_idx == 0:
        ax.set_ylabel('ΔNLL per token removed', fontsize=11)
        ax.legend(fontsize=9)

fig.suptitle('Dose Response: Span Size 128 vs 256\n'
             'Top: Raw ΔNLL (bigger span → bigger hit expected)\n'
             'Bottom: Per-token ΔNLL (intrinsic informativeness)', 
             fontsize=13, fontweight='bold', y=1.02)

plt.tight_layout()
plt.savefig(output_dir / 'dose_response.png', dpi=150, bbox_inches='tight')
plt.show()

# Print the comparison table
print("\n" + "="*70)
print("DOSE RESPONSE: Per-Token Influence (ΔNLL / tokens_removed)")
print("="*70)
print("\\nIf per-token values are SIMILAR across span sizes, effect scales with size.")
print("If per-token is HIGHER for 256, there's compounding/overlap effect.\n")

for region in REGION_ORDER:
    print(f"--- {region} ---")
    print(f"{'Span':<25} {'128 raw':>10} {'256 raw':>10} {'128 /tok':>10} {'256 /tok':>10}")
    print("-"*70)
    
    for span in COMMON_SPANS:
        raw_128 = df[(df['span_size']==128) & (df['span_label']==span) & (df['score_region']==region)]['delta_nll'].mean()
        raw_256 = df[(df['span_size']==256) & (df['span_label']==span) & (df['score_region']==region)]['delta_nll'].mean()
        pt_128 = df[(df['span_size']==128) & (df['span_label']==span) & (df['score_region']==region)]['delta_nll_per_token'].mean()
        pt_256 = df[(df['span_size']==256) & (df['span_label']==span) & (df['score_region']==region)]['delta_nll_per_token'].mean()
        print(f"{span:<25} {raw_128:>10.4f} {raw_256:>10.4f} {pt_128:>10.6f} {pt_256:>10.6f}")
    print()

## 8a. Per-Token Influence Analysis: Near vs Far

Does the "second half" (immediate_pre_target) dominate per-token influence?
Compute `near_minus_far = delta_nll_per_token(immediate_pre) - delta_nll_per_token(early)`

In [ ]:
# Check if "second half" dominates per-token influence
# Compare early vs immediate_pre_target delta_nll_per_token

print("="*80)
print("PER-TOKEN INFLUENCE: Does immediate context dominate?")
print("="*80)

# Focus on span_size=128 for fair comparison (both spans exist)
df_128 = df[df['span_size'] == 128].copy()

# Extract early and immediate_pre_target
early_data = df_128[df_128['span_label'] == 'early'][['essay_id', 'score_region', 'delta_nll_per_token']].copy()
early_data.columns = ['essay_id', 'score_region', 'per_token_early']

imm_pre_data = df_128[df_128['span_label'] == 'immediate_pre_target'][['essay_id', 'score_region', 'delta_nll_per_token']].copy()
imm_pre_data.columns = ['essay_id', 'score_region', 'per_token_imm_pre']

# Merge to compute near_minus_far per essay
per_token_comparison = early_data.merge(imm_pre_data, on=['essay_id', 'score_region'])

# Add essay metadata
essay_meta = df_128[['essay_id', 'score_bin', 'grade', 'baseline_nll', 'token_count']].drop_duplicates()
per_token_comparison = per_token_comparison.merge(essay_meta, on='essay_id')

# Compute near_minus_far
per_token_comparison['near_minus_far'] = per_token_comparison['per_token_imm_pre'] - per_token_comparison['per_token_early']

print(f"\nPer-token comparison dataset: {len(per_token_comparison)} rows")
print(f"\nOverall means:")
print(f"  early per-token:         {per_token_comparison['per_token_early'].mean():.6f}")
print(f"  immediate_pre per-token: {per_token_comparison['per_token_imm_pre'].mean():.6f}")
print(f"  near_minus_far:          {per_token_comparison['near_minus_far'].mean():.6f}")

# Positive near_minus_far means immediate context is more informative per token
pct_positive = (per_token_comparison['near_minus_far'] > 0).mean() * 100
print(f"\n  % essays where immediate_pre > early (per token): {pct_positive:.1f}%")

# By score region
print("\n" + "-"*60)
print("By scoring region:")
for region in REGION_ORDER:
    subset = per_token_comparison[per_token_comparison['score_region'] == region]
    pct_pos = (subset['near_minus_far'] > 0).mean() * 100
    print(f"  {region}: near_minus_far = {subset['near_minus_far'].mean():.6f} ({pct_pos:.1f}% positive)")

In [ ]:
# Group comparison: delta_nll_per_token for early vs immediate_pre_target
print("="*80)
print("GROUP COMPARISON: Per-Token Influence by Score Bin")
print("="*80)

print("\n--- early span (tokens 0-128) ---")
for region in REGION_ORDER:
    print(f"\n{region}:")
    for score_bin in GROUP_ORDER:
        subset = per_token_comparison[(per_token_comparison['score_bin'] == score_bin) & 
                                       (per_token_comparison['score_region'] == region)]
        mean = subset['per_token_early'].mean()
        sem = subset['per_token_early'].sem()
        print(f"  {score_bin}: {mean:.6f} ± {1.96*sem:.6f}")

print("\n--- immediate_pre_target span (tokens 384-512) ---")
for region in REGION_ORDER:
    print(f"\n{region}:")
    for score_bin in GROUP_ORDER:
        subset = per_token_comparison[(per_token_comparison['score_bin'] == score_bin) & 
                                       (per_token_comparison['score_region'] == region)]
        mean = subset['per_token_imm_pre'].mean()
        sem = subset['per_token_imm_pre'].sem()
        print(f"  {score_bin}: {mean:.6f} ± {1.96*sem:.6f}")

print("\n--- near_minus_far (immediate_pre - early, per token) ---")
for region in REGION_ORDER:
    print(f"\n{region}:")
    for score_bin in GROUP_ORDER:
        subset = per_token_comparison[(per_token_comparison['score_bin'] == score_bin) & 
                                       (per_token_comparison['score_region'] == region)]
        mean = subset['near_minus_far'].mean()
        sem = subset['near_minus_far'].sem()
        pct_pos = (subset['near_minus_far'] > 0).mean() * 100
        print(f"  {score_bin}: {mean:+.6f} ± {1.96*sem:.6f} ({pct_pos:.0f}% positive)")

In [ ]:
# REGRESSION: near_minus_far ~ score_bin + controls
# Does near_minus_far differ by score_bin after controlling for fluency/length?

print("="*80)
print("REGRESSION: near_minus_far ~ C(score_bin) + controls")
print("="*80)

# Standardize controls
per_token_comparison['baseline_nll_z'] = (per_token_comparison['baseline_nll'] - per_token_comparison['baseline_nll'].mean()) / per_token_comparison['baseline_nll'].std()
per_token_comparison['token_count_z'] = (per_token_comparison['token_count'] - per_token_comparison['token_count'].mean()) / per_token_comparison['token_count'].std()

# Full model
formula_nmf = 'near_minus_far ~ C(score_bin) + C(score_region) + baseline_nll_z + token_count_z'
model_nmf = smf.ols(formula_nmf, data=per_token_comparison).fit()

print(f"\nFormula: {formula_nmf}")
print(f"R²: {model_nmf.rsquared:.4f}, Adj R²: {model_nmf.rsquared_adj:.4f}, n={int(model_nmf.nobs)}")

print("\nKey coefficients:")
for param in ['Intercept', 'C(score_bin)[T.mid]', 'C(score_bin)[T.high]', 
              'C(score_region)[T.score_mid]', 'C(score_region)[T.score_late]',
              'baseline_nll_z', 'token_count_z']:
    if param in model_nmf.params:
        coef = model_nmf.params[param]
        pval = model_nmf.pvalues[param]
        sig = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
        print(f"  {param:<35}: β={coef:+.6f}, p={pval:.4f} {sig}")

print("\nINTERPRETATION:")
print("  Positive near_minus_far = immediate context more informative per token")
print("  Negative near_minus_far = early context more informative per token")
print("  If high-scorers have LOWER near_minus_far, they rely more on distributed context")

In [ ]:
# VISUALIZATION: near_minus_far by score bin
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Bar chart of near_minus_far by score_bin and score_region
ax = axes[0]
x_positions = np.arange(len(REGION_ORDER))
width = 0.25

for i, score_bin in enumerate(GROUP_ORDER):
    means = []
    cis = []
    for region in REGION_ORDER:
        subset = per_token_comparison[(per_token_comparison['score_bin'] == score_bin) & 
                                       (per_token_comparison['score_region'] == region)]
        means.append(subset['near_minus_far'].mean())
        cis.append(1.96 * subset['near_minus_far'].sem())
    
    offset = (i - 1) * width
    ax.bar(x_positions + offset, means, width, yerr=cis,
           label=score_bin, color=SCORE_COLORS[score_bin], capsize=3, alpha=0.8)

ax.axhline(0, color='black', linestyle='--', linewidth=0.8, alpha=0.5)
ax.set_xticks(x_positions)
ax.set_xticklabels(['score_early\n(512-576)', 'score_mid\n(576-640)', 'score_late\n(640-768)'])
ax.set_ylabel('near_minus_far\n(per-token: immediate_pre − early)', fontsize=10)
ax.set_title('Context Localization Index by Score Bin\n(Positive = immediate context dominates per token)', 
             fontsize=11, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3, axis='y')

# Right: Distribution of near_minus_far by score_bin (collapsed across regions)
ax = axes[1]
for score_bin in GROUP_ORDER:
    subset = per_token_comparison[per_token_comparison['score_bin'] == score_bin]['near_minus_far']
    ax.hist(subset, bins=30, alpha=0.5, label=score_bin, color=SCORE_COLORS[score_bin], density=True)

ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('near_minus_far', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Distribution of Context Localization Index\n(Left of 0 = early context dominates)', 
             fontsize=11, fontweight='bold')
ax.legend(title='Score Bin')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(output_dir / 'near_minus_far_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nSaved: {output_dir}/near_minus_far_analysis.png")

In [ ]:
# Save near_minus_far per-essay results
per_token_comparison.to_csv(output_dir / 'near_minus_far_per_essay.csv', index=False)
print(f"Saved: {output_dir}/near_minus_far_per_essay.csv ({len(per_token_comparison)} rows)")

## 9. Save Results

In [ ]:
# Save full results
df.to_csv(output_dir / 'span_ablation_results.csv', index=False)

# Save summary
df_summary.to_csv(output_dir / 'span_ablation_summary.csv', index=False)

# Save pivot tables
pivot_raw.to_csv(output_dir / 'span_ablation_pivot_raw.csv')
pivot_norm.to_csv(output_dir / 'span_ablation_pivot_normalized.csv')

# Save regression results (including near_minus_far)
with open(output_dir / 'regression_span_ablation.txt', 'w') as f:
    f.write("ENHANCED SPAN ABLATION REGRESSION ANALYSIS\n")
    f.write("="*80 + "\n\n")
    f.write(f"Configuration:\n")
    f.write(f"  Cohort: score_long (50/50/50, all ≥778 tokens, all EXTENDED)\n")
    f.write(f"  Span sizes (dose): {SPAN_SIZES}\n")
    f.write(f"  Span locations: disjoint tiling (128: 4 spans, 256: 2 spans) + random\n")
    f.write(f"  Scoring regions: {SCORE_REGIONS}\n")
    f.write(f"  Random samples per essay: {N_RANDOM_SPANS}\n")
    f.write(f"  Random seed: {RANDOM_SEED}\n")
    f.write("\n" + "="*80 + "\n\n")
    f.write("PRIMARY METRICS:\n")
    f.write("  1. ΔNLL (ablated - baseline): predictive support from span\n")
    f.write("  2. Rank displacement (mean/median): how much true token rank worsens\n")
    f.write("  3. Normalized influence: within-essay proportion\n")
    f.write("  4. delta_nll_per_token: ΔNLL / span_size (intrinsic informativeness)\n")
    f.write("  5. near_minus_far: per_token(immediate_pre) - per_token(early)\n")
    f.write("\n" + "="*80 + "\n\n")
    f.write("FULL MODEL (span_size=128):\n")
    f.write(f"Formula: {formula}\n")
    f.write(model_full.summary().as_text())
    f.write("\n\n" + "="*80 + "\n\n")
    f.write("NORMALIZED INFLUENCE MODEL:\n")
    f.write(f"Formula: {formula_norm}\n")
    f.write(model_norm.summary().as_text())
    f.write("\n\n" + "="*80 + "\n\n")
    f.write("RAW ΔNLL WITH CONTROLS:\n")
    f.write(f"Formula: {formula_raw}\n")
    f.write(model_raw.summary().as_text())
    f.write("\n\n" + "="*80 + "\n\n")
    f.write("RANK DISPLACEMENT WITH CONTROLS:\n")
    f.write(f"Formula: {formula_rank}\n")
    f.write(model_rank.summary().as_text())
    f.write("\n\n" + "="*80 + "\n")
    f.write("NEAR_MINUS_FAR ANALYSIS\n")
    f.write("="*80 + "\n\n")
    f.write("near_minus_far = delta_nll_per_token(immediate_pre) - delta_nll_per_token(early)\n")
    f.write("Interpretation:\n")
    f.write("  Positive = immediate context more informative per token (localized)\n")
    f.write("  Negative = early context more informative per token (distributed)\n\n")
    f.write(f"Formula: {formula_nmf}\n")
    f.write(model_nmf.summary().as_text())

print(f"\nSaved to {output_dir}/")
print(f"  - span_ablation_results.csv ({len(df)} rows)")
print(f"  - span_ablation_summary.csv")
print(f"  - span_ablation_pivot_raw.csv")
print(f"  - span_ablation_pivot_normalized.csv")
print(f"  - near_minus_far_per_essay.csv")
print(f"  - regression_span_ablation.txt")
print(f"  - influence_heatmaps.png")
print(f"  - influence_heatmaps_by_scorebin.png")
print(f"  - influence_by_group.png")
print(f"  - decay_analysis.png")
print(f"  - dose_response.png")
print(f"  - near_minus_far_analysis.png")

In [ ]:
# Final summary
print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print("""
GOAL: Measure influence geometry - whether influence is distributed vs localized,
      and whether high-quality essays redistribute influence.

PRIMARY METRICS:
  1. ΔNLL = ablated NLL - baseline NLL
     → Higher = more predictive support came from that span
  2. Rank displacement = ablated rank - baseline rank
     → Higher = true token rank worsened more
  3. Normalized influence = span ΔNLL / total ΔNLL
     → Shows WHERE influence comes from (proportion)
  4. delta_nll_per_token = ΔNLL / span_size
     → Fair cross-dose comparison (intrinsic informativeness)
  5. near_minus_far = per_token(immediate_pre) - per_token(early)
     → Single number for context localization
     → Positive = immediate context dominates per token
     → Negative = early context more informative per token

KEY QUESTIONS:
  1. Does boundary-adjacent context (immediate_pre_target) hit score_early hardest?
  2. Does effect decay with distance (score_early > score_mid > score_late)?
  3. Do score groups differ in WHERE influence comes from?
  4. Is pattern robust to dose (span size 128 vs 256)?
  5. Does near_minus_far differ by score_bin? (distributed vs localized reliance)

FILES SAVED:
  - span_ablation_results.csv (essay × span × region)
  - span_ablation_summary.csv (group means + CI)
  - span_ablation_pivot_raw/normalized.csv
  - near_minus_far_per_essay.csv (context localization index)
  - regression_span_ablation.txt (all models including near_minus_far)
  - influence_heatmaps.png (aggregated)
  - influence_heatmaps_by_scorebin.png (split by low/mid/high)
  - influence_by_group.png
  - decay_analysis.png
  - dose_response.png
  - near_minus_far_analysis.png
""")